In [1]:
import pandas as pd
import os
from glob import glob
from pathlib import Path
import numpy as np

In [2]:
# Helper

def clean_county(series):
    return (
        series.astype(str)
        .str.replace(", West Virginia", "", regex=False)
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.title()
    )

In [3]:
# Labor Data (LAUS)

labor_folder = r"C:\Users\mpenk\Desktop\wv_project\labor"
labor_files = glob(os.path.join(labor_folder, "*.xlsx"))

fips = pd.read_csv("county_fips.csv")

labor_list = []

for file in labor_files:
    df = pd.read_excel(file, skiprows=1)

    df[['county', 'state']] = df['County Name/State Abbreviation'].str.split(', ', expand=True)
    df = df[df['state'] == 'WV'].copy()

    df = df.drop(columns=[
        'LAUS Code',
        'State FIPS Code',
        'County FIPS Code',
        'County Name/State Abbreviation',
        'state'
    ])

    df.columns = (
        df.columns
        .str.lower()
        .str.replace(r'[^\w\s]', '', regex=True)
        .str.strip()
        .str.replace(r'\s+', '_', regex=True)
    )

    df = df.astype({
        'year': 'int64',
        'labor_force': 'int64',
        'employed': 'int64',
        'unemployed': 'int64',
        'unemployment_rate': 'float64',
        'county': 'string'
    })

    df['county'] = clean_county(df['county'])

    df = df.merge(fips, on='county', how='left')

    df = df[[
        'county', 'county_fips', 'year',
        'labor_force', 'employed', 'unemployed', 'unemployment_rate'
    ]]

    labor_list.append(df)

labor_df = pd.concat(labor_list, ignore_index=True)

In [4]:
# Demo Data (ACS)

demo_folder = Path(r"C:\Users\mpenk\Desktop\wv_project\demo")

demo_list = []

for file in demo_folder.glob("*Data.csv"):

    year = int(file.name.split("ACSST5Y")[1].split(".")[0])
   

    df = pd.read_csv(file, header=0, skiprows=[1])

    # Column Fix Only
    if year <= 2016:
        df = df[[
            "NAME",
            "S0101_C01_001E",
            "S0101_C01_024E",
            "S0101_C01_028E"
        ]]

        df = df.rename(columns={
            "NAME": "county",
            "S0101_C01_001E": "total_pop",
            "S0101_C01_024E": "age_16_plus",
            "S0101_C01_028E": "age_65_plus"
        })

    else:  # 2017+
        df = df[[
            "NAME",
            "S0101_C01_001E",
            "S0101_C02_025E",
            "S0101_C01_030E"
        ]]

        df = df.rename(columns={
            "NAME": "county",
            "S0101_C01_001E": "total_pop",
            "S0101_C02_025E": "age_16_plus",
            "S0101_C01_030E": "age_65_plus"
        })

    df['county'] = clean_county(df['county'])

    cols = ["total_pop", "age_16_plus", "age_65_plus"]
    df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

    df["year"] = year

    demo_list.append(df)

demo_df = pd.concat(demo_list, ignore_index=True)

demo_df = demo_df[
    ~demo_df["county"].str.contains("United States", na=False)
]

demo_df = demo_df.sort_values(["county", "year"]).reset_index(drop=True)

# Merge

merged_df = labor_df.merge(
    demo_df,
    on=["county", "year"],
    how="left"
)

# Final

merged_df = merged_df[[
    "county",
    "county_fips",
    "year",
    "labor_force",
    "employed",
    "unemployed",
    "unemployment_rate",
    "total_pop",
    "age_16_plus",
    "age_65_plus"
]].sort_values(["county", "year"]).reset_index(drop=True)

merged_df.head()

,county,county_fips,year,labor_force,employed,unemployed,unemployment_rate,total_pop,age_16_plus,age_65_plus
0,Barbour County,54001,2010,6868,6194,674,9.8,16256,80.8,16.2
1,Barbour County,54001,2011,6774,6165,609,9.0,16373,80.7,16.2
2,Barbour County,54001,2012,6736,6197,539,8.0,16472,80.6,16.8
3,Barbour County,54001,2013,6730,6255,475,7.1,16655,81.0,17.2
4,Barbour County,54001,2014,6904,6447,457,6.6,16723,81.1,17.8


In [5]:
merged_df[["year","age_16_plus","age_65_plus","total_pop"]].sample(20)

,year,age_16_plus,age_65_plus,total_pop
542,2012,82.1,16.4,7630
715,2020,83.0,1953.0,8736
476,2021,81.9,3038.0,12492
818,2018,82.1,4256.0,21711
306,2016,81.2,19.5,16422
566,2021,84.7,2061.0,8006
274,2014,79.2,12.9,54650
808,2023,81.5,17791.0,83829
55,2020,82.7,3220.0,14032
241,2011,80.7,16.5,68745


In [6]:
merged_df.groupby("year")["age_16_plus"].mean()

year
2010    81.514545
2011    81.701818
2012    81.752727
2013    81.921818
2014    81.974545
2015    82.061818
2016    82.116364
2017    82.194545
2018    82.323636
2019    82.445455
2020    82.610909
2021    82.547273
2022    82.810909
2023    82.863636
2024    82.980000
Name: age_16_plus, dtype: float64

In [7]:
merged_df["age_65_plus"] = np.where(
    merged_df["age_65_plus"] > 100,
    (merged_df["age_65_plus"] / merged_df["total_pop"]) * 100,
    merged_df["age_65_plus"]
)

In [8]:
merged_df[["year","age_16_plus","age_65_plus","total_pop"]].sample(20)

,year,age_16_plus,age_65_plus,total_pop
255,2010,80.5,17.300000,29084
268,2023,81.2,20.974309,27753
544,2014,83.0,17.400000,7600
695,2015,84.6,22.000000,6972
314,2024,80.7,21.075269,16740
222,2022,83.8,24.302072,28907
331,2011,81.8,15.000000,36564
585,2010,78.9,14.000000,54940
397,2017,81.8,19.089664,19707
785,2015,81.0,17.500000,5841


In [25]:
# Logical checks block

In [20]:
assert (merged_df["age_16_plus"] <= 100).all()
assert (merged_df["age_65_plus"] <= 100).all()
assert (merged_df["age_65_plus"] <= merged_df["age_16_plus"]).all()

In [21]:
merged_df["age_16_plus_pop"] = (
    merged_df["age_16_plus"] / 100
) * merged_df["total_pop"]

merged_df["age_65_plus_pop"] = (
    merged_df["age_65_plus"] / 100
) * merged_df["total_pop"]

In [22]:
assert (merged_df["labor_force"] <= merged_df["age_16_plus_pop"]).all()
assert (merged_df["labor_force"] >= 0).all()

In [23]:
merged_df["lfpr"] = (
    merged_df["labor_force"] / merged_df["age_16_plus_pop"]
) * 100

merged_df["not_in_labor_force"] = (
    merged_df["age_16_plus_pop"] - merged_df["labor_force"]
)

In [24]:
assert ((merged_df["lfpr"] >= 0) & (merged_df["lfpr"] <= 100)).all()
assert (merged_df["not_in_labor_force"] >= 0).all()

In [26]:
assert (
    (merged_df["age_16_plus_pop"] / merged_df["total_pop"])
    .between(0.6, 0.95)
).all()

In [29]:
merged_df.head()

,county,county_fips,year,labor_force,employed,unemployed,unemployment_rate,total_pop,age_16_plus,age_65_plus,age_16_plus_pop,age_65_plus_pop,lfpr,not_in_labor_force
0,Barbour County,54001,2010,6868,6194,674,9.8,16256,80.8,16.2,13134.848,2633.472,52.288386,6266.848
1,Barbour County,54001,2011,6774,6165,609,9.0,16373,80.7,16.2,13213.011,2652.426,51.267648,6439.011
2,Barbour County,54001,2012,6736,6197,539,8.0,16472,80.6,16.8,13276.432,2767.296,50.736523,6540.432
3,Barbour County,54001,2013,6730,6255,475,7.1,16655,81.0,17.2,13490.550,2864.660,49.886773,6760.550
4,Barbour County,54001,2014,6904,6447,457,6.6,16723,81.1,17.8,13562.353,2976.694,50.905621,6658.353


In [30]:
merged_df.to_csv("fact_labor_demo.csv", index = False)